In [0]:

%pip install   lightgbm
# Databricks notebook source
# Databricks notebook source
# Databricks notebook source
#

In [0]:
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # LightGBM Stock Close-Price Prediction Pipeline (Experimental Global Model)
# MAGIC
# MAGIC **الفكرة:** بدل ما ندرب LSTM منفصل لكل شركة (500 موديل)، بندرب **موديل واحد عام**
# MAGIC على كل الشركات مع بعض، وبنضيف `ticker` كـ categorical feature عشان الموديل يتعلم
# MAGIC الفروق بين الشركات من جوه البيانات نفسها، مش عن طريق عزل كل شركة في موديل مستقل.
# MAGIC
# MAGIC **الفرق عن نسخة الـ LSTM:**
# MAGIC - موديل واحد بس بدل 500 (وقت تدريب أقل بكتير)
# MAGIC - مفيش MinMaxScaler ولا مشكلة extrapolation (الأشجار بترجع قيم من نطاق التدريب نفسه)
# MAGIC - مفيش batching/incremental writes معقدة، التدريب بيحصل مرة واحدة
# MAGIC - نفس شكل جدول الـ Gold بالظبط (نفس الأعمدة) عشان تقارن النتايج بسهولة

# COMMAND ----------

# MAGIC %pip install lightgbm

# COMMAND ----------

import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "finance_intelligence_hub"
LOOKBACK_MONTHS = 6              # نفس المدة المستخدمة في نسخة الـ LSTM المصححة
TOP_N_TICKERS = 500
LAG_DAYS = [1, 2, 3, 5, 10]       # أعمدة lag هتتضاف كـ features (بديل الـ WINDOW_SIZE بتاع LSTM)
ROLLING_WINDOWS = [5, 10]        # متوسطات متحركة كـ features

GOLD_TABLE = f"{CATALOG}.gold.stock_price_predictions"          # نفس جدول الـ LSTM
GOLD_TABLE_LGBM = f"{CATALOG}.gold.stock_price_predictions"  # جدول تجريبي منفصل، بنفس الشكل
                                                                    # بالظبط، عشان منكتبش فوق نتايج
                                                                    # الـ LSTM ونقدر نقارن الاتنين

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Top companies by Dollar Volume (نفس منطق نسخة الـ LSTM)

# COMMAND ----------

top_tickers_df = spark.sql(f"""
    SELECT ticker
    FROM (
        SELECT
            ticker,
            price * volume AS dollar_vol
        FROM {CATALOG}.silver.companies
        WHERE price IS NOT NULL AND volume IS NOT NULL
    )
    ORDER BY dollar_vol DESC
    LIMIT {TOP_N_TICKERS}
""")

top_tickers = [row.ticker for row in top_tickers_df.collect()]
print(f"✅ Number of companies selected: {len(top_tickers)}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Last 6 months of stock prices + daily sentiment (نفس الخطوات 2+3 في نسخة الـ LSTM)

# COMMAND ----------

six_months_ago = F.date_sub(F.current_date(), LOOKBACK_MONTHS * 30)

stock_prices_df = (
    spark.table(f"{CATALOG}.silver.stock_prices")
    .filter(F.col("ticker").isin(top_tickers))
    .filter(F.col("trade_date") >= six_months_ago)
    .select(
        "ticker", "trade_date", "open_price", "high_price",
        "low_price", "close_price", "adjusted_close_price", "volume"
    )
)

daily_sentiment_df = (
    spark.table(f"{CATALOG}.silver.news_sentiments")
    .filter(F.col("ticker").isin(top_tickers))
    .withColumn("pub_date", F.to_date("published_at"))
    .groupBy("ticker", "pub_date")
    .agg(
        F.avg("positive_score").alias("avg_positive_score"),
        F.avg("negative_score").alias("avg_negative_score"),
        F.avg("neutral_score").alias("avg_neutral_score"),
        F.count("article_id").alias("news_count"),
    )
)

merged_df = (
    stock_prices_df.alias("sp")
    .join(
        daily_sentiment_df.alias("sent"),
        (F.col("sp.ticker") == F.col("sent.ticker"))
        & (F.col("sp.trade_date") == F.col("sent.pub_date")),
        "left",
    )
    .select(
        F.col("sp.ticker").alias("ticker"),
        F.col("sp.trade_date").alias("trade_date"),
        "open_price", "high_price", "low_price",
        "close_price", "adjusted_close_price", "volume",
        F.coalesce("avg_positive_score", F.lit(0.0)).alias("avg_positive_score"),
        F.coalesce("avg_negative_score", F.lit(0.0)).alias("avg_negative_score"),
        F.coalesce("avg_neutral_score", F.lit(0.0)).alias("avg_neutral_score"),
        F.coalesce("news_count", F.lit(0)).alias("news_count"),
    )
)

print(f"✅ Merged rows: {merged_df.count()}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. تجميع كل البيانات في pandas واحد (موديل عام، مش لكل شركة لوحدها)
# MAGIC
# MAGIC هنا الفرق الجوهري عن نسخة الـ LSTM: مفيش loop على batches ولا applyInPandas لكل
# MAGIC ticker — بنجيب كل البيانات مرة واحدة كـ pandas DataFrame واحد (حجم البيانات هنا
# MAGIC معقول: ~500 شركة × ~130 يوم تداول، مش ملايين الصفوف) وندرب موديل واحد عليها كلها.

# COMMAND ----------

full_pdf = merged_df.orderBy("ticker", "trade_date").toPandas()
print(f"✅ Total rows collected to pandas: {len(full_pdf)}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. هندسة الـ Features (بديل الـ sliding window بتاع LSTM)
# MAGIC
# MAGIC بدل ما ندي الموديل نافذة خام من آخر 10 أيام (زي LSTM)، بنحول التاريخ الزمني لأعمدة
# MAGIC صريحة: lag values + rolling averages + `ticker` نفسه كـ categorical feature.
# MAGIC ده اللي بيخلي موديل واحد قادر يفرّق بين الشركات المختلفة.

# COMMAND ----------

def add_features(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf = pdf.sort_values(["ticker", "trade_date"]).copy()
    grp = pdf.groupby("ticker")

    # Lag features لكل feature أساسي
    base_cols = [
        "close_price", "adjusted_close_price", "volume",
        "avg_positive_score", "avg_negative_score", "avg_neutral_score", "news_count",
    ]
    for col in base_cols:
        for lag in LAG_DAYS:
            pdf[f"{col}_lag{lag}"] = grp[col].shift(lag)

    # Rolling averages (متحسوبة من غير تسريب: shift(1) الأول عشان ميدخلش اليوم الحالي)
    for col in ["close_price", "volume"]:
        for w in ROLLING_WINDOWS:
            pdf[f"{col}_roll_mean{w}"] = (
                grp[col].shift(1).rolling(w).mean().reset_index(level=0, drop=True)
            )

    # الهدف: close_price اليوم التالي لكل شركة (المتغير اللي هنتوقعه)
    pdf["target_next_close"] = grp["close_price"].shift(-1)

    return pdf


featured_pdf = add_features(full_pdf)
print(f"✅ Feature columns added. Shape: {featured_pdf.shape}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. تقسيم Train / Validation زمني (بدون shuffle، زي نسخة الـ LSTM)
# MAGIC
# MAGIC نفس فكرة الـ LSTM: آخر جزء من الزمن بيتاخد validation، مفيش شفل عشوائي لأنها time series.
# MAGIC كمان بنفصل آخر صف حقيقي لكل شركة (اللي هنعمله عليه التوقع الفعلي، مفيهوش target حقيقي
# MAGIC لسه لأنه بيمثل "بكرة").

# COMMAND ----------

feature_cols = [c for c in featured_pdf.columns if (
    c.endswith(tuple(f"_lag{l}" for l in LAG_DAYS)) or "_roll_mean" in c
)]

# الصف الأخير لكل شركة = الصف اللي هنعمله عليه التوقع الحقيقي (مفيهوش target حقيقي)
last_rows = featured_pdf.groupby("ticker").tail(1).copy()

# باقي الصفوف (اللي عندها target حقيقي) هي بيانات التدريب/التقييم
train_eval_pdf = featured_pdf.dropna(subset=feature_cols + ["target_next_close"]).copy()

# تقسيم زمني: آخر 15% من كل تاريخ البيانات (مش لكل شركة، عشان يفضل التقسيم متسق زمنيًا
# على مستوى كل الداتاسيت، زي VALIDATION_FRACTION بتاعت نسخة الـ LSTM)
cutoff_date = train_eval_pdf["trade_date"].quantile(0.85, interpolation="nearest")
train_pdf = train_eval_pdf[train_eval_pdf["trade_date"] < cutoff_date]
val_pdf = train_eval_pdf[train_eval_pdf["trade_date"] >= cutoff_date]

print(f"✅ Train rows: {len(train_pdf)} | Validation rows: {len(val_pdf)} | Predict rows: {len(last_rows)}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. تدريب موديل LightGBM واحد على كل الشركات مع بعض

# COMMAND ----------

import lightgbm as lgb

train_pdf = train_pdf.copy()
val_pdf = val_pdf.copy()
last_rows = last_rows.copy()

# ticker كـ categorical feature — ده اللي بيخلي الموديل الواحد يفرّق بين الشركات
train_pdf["ticker"] = train_pdf["ticker"].astype("category")
val_pdf["ticker"] = val_pdf["ticker"].astype("category")
last_rows["ticker"] = pd.Categorical(
    last_rows["ticker"], categories=train_pdf["ticker"].cat.categories
)

lgbm_feature_cols = feature_cols + ["ticker"]

train_set = lgb.Dataset(
    train_pdf[lgbm_feature_cols], label=train_pdf["target_next_close"],
    categorical_feature=["ticker"],
)
val_set = lgb.Dataset(
    val_pdf[lgbm_feature_cols], label=val_pdf["target_next_close"],
    categorical_feature=["ticker"], reference=train_set,
)

params = {
    "objective": "regression",
    "metric": "mae",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 20,
    "verbose": -1,
}

model = lgb.train(
    params,
    train_set,
    num_boost_round=500,
    valid_sets=[train_set, val_set],
    valid_names=["train", "val"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=30),   # بديل EarlyStopping(monitor="val_loss") في LSTM
        lgb.log_evaluation(period=0),
    ],
)

print(f"✅ Training done. Best iteration: {model.best_iteration}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. التوقع النهائي لكل شركة + كتابة النتيجة بنفس شكل جدول الـ Gold بتاع LSTM

# COMMAND ----------

predicted = model.predict(
    last_rows[lgbm_feature_cols], num_iteration=model.best_iteration
)
last_rows["predicted_close_price"] = np.round(predicted, 2)
last_rows["last_known_close_price"] = last_rows["close_price"]
last_rows["last_trade_date"] = last_rows["trade_date"]
last_rows["train_rows_used"] = float(len(train_pdf))

predictions_pdf = last_rows[[
    "ticker", "last_trade_date", "predicted_close_price",
    "last_known_close_price", "train_rows_used",
]]

predictions_df = spark.createDataFrame(predictions_pdf)

companies_df = spark.table(f"{CATALOG}.silver.companies")

gold_df = (
    companies_df.alias("c")
    .join(predictions_df.alias("p"), "ticker", "inner")
    .select(
        "c.*",
        F.col("p.last_trade_date"),
        F.col("p.predicted_close_price"),
        F.col("p.last_known_close_price"),
        F.col("p.train_rows_used"),
        F.current_timestamp().alias("prediction_generated_at"),
    )
)

(
    gold_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(GOLD_TABLE_LGBM)
)

print(f"\n✅ Done! Predictions for {gold_df.count()} companies written to: {GOLD_TABLE_LGBM}")
print(f"(نسخة الـ LSTM لسه موجودة في: {GOLD_TABLE} — تقدر تقارن الاتنين)")
display(spark.table(GOLD_TABLE_LGBM).limit(20))